In [1]:
import numpy as np
import spacy
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load spaCy
nlp = spacy.load("en_core_web_sm")

# Tiny example training dataset
texts = [
    "I feel happy today",
    "I am feeling great",
    "I feel sad",
    "I am very unhappy",
    "I am stressed",
    "I feel worried",
    "I feel calm",
    "I am relaxed"
]

# Labels: 0 = happy, 1 = sad, 2 = stressed, 3 = calm
labels = np.array([0, 0, 1, 1, 2, 2, 3, 3])

# Text preprocessing using spaCy
clean_texts = []

for text in texts:
    doc = nlp(text.lower())
    tokens = [
        token.lemma_
        for token in doc
        if not token.is_punct and not token.is_space
    ]
    clean_texts.append(" ".join(tokens))

# Convert words into numerical sequences
tokenizer = Tokenizer()
tokenizer.fit_on_texts(clean_texts)

sequences = tokenizer.texts_to_sequences(clean_texts)
X = pad_sequences(sequences, padding="post", maxlen=6)

# Build an LSTM model
model = Sequential([
    Embedding(
        input_dim=len(tokenizer.word_index) + 1,
        output_dim=16,
        input_length=6
    ),
    LSTM(16),
    Dense(4, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Train the demonstration model
model.fit(X, labels, epochs=100, verbose=0)

# Predict the emotion of a new message
message = "I feel worried today"

doc = nlp(message.lower())
tokens = [
    token.lemma_
    for token in doc
    if not token.is_punct and not token.is_space
]

sequence = tokenizer.texts_to_sequences(
    [" ".join(tokens)]
)
padded = pad_sequences(sequence, padding="post", maxlen=6)

prediction = model.predict(padded, verbose=0)
emotion_id = int(np.argmax(prediction))

emotions = ["happy", "sad", "stressed", "calm"]

responses = {
    "happy": "It is nice to hear that you are feeling happy!",
    "sad": "I am sorry you are feeling sad. Would you like to talk about it?",
    "stressed": "I understand that things may feel difficult. Try taking a slow breath.",
    "calm": "I am glad you are feeling calm. Take care of yourself!"
}

print("User message:", message)
print("Predicted emotion:", emotions[emotion_id])
print("Chatbot response:", responses[emotions[emotion_id]])


/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


User message: I feel worried today
Predicted emotion: happy
Chatbot response: It is nice to hear that you are feeling happy!
